In [27]:
import sys
from pathlib import Path

from scipy.optimize import minimize

import pprint
from matplotlib import font_manager
import matplotlib.pyplot as plt
# Add the path to the downloaded fonts
font_dirs = ['/home/simoneponcioni/Documents/99_OTHERS/my_fonts/']  # Replace with the actual path to your fonts
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)

for font_file in font_files:
    font_manager.fontManager.addfont(font_file)

# Load the style file
plt.style.use('../../01_CODE/src/style/paper-style.mplstyle')

# Set up fonts
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Outfit', 'Cabin']

sys.path.append(str(Path('/home/simoneponcioni/Documents/01_PHD/03_Methods/HR-pQCT_database/01_CODE/src')))
import statistics_hrpqct as statistics_hrpqct
import dataclasses_hrpqct as dataclass_hrpqct
from datetime import datetime
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
from scipy.stats import ttest_ind

In [28]:
# Import raw datafile from csv raw filepath#
dir_path = Path('../../00_DB/')
# df_path = 'HR-pQCT_database_expanded_2025-05-12_14-53.csv'
df_path = 'HR-pQCT_database_expanded_2025-06-16_16-18.csv'
dataset_path = dir_path / df_path

In [29]:
# Instantiate class Statistics for the nodaratis database#
name_combined = 'Combined'
name_originator = 'POS'

common_df = pd.read_csv(dataset_path, sep=',')
dataset = dataclass_hrpqct.HRpQCT_Dataset(df=common_df, hfe_expansion=True)

# extract only dataclass for one patient, it will determine which sex is the patient for further analysis
patient_UID = 4
df_patient = common_df[common_df["UID"] == patient_UID]
dataset_patient = dataclass_hrpqct.HRpQCT_Dataset(df=df_patient, hfe_expansion=True)
print(dataset_patient.Gender)
db_stats = statistics_hrpqct.Statistics(df=dataset.df, df_patient=dataset_patient.df, name='HRpQCT_common_database', originator='POS', savefig=False, showfig=False)

3    Female
Name: Gender, dtype: object


In [30]:
patient_gender = dataset_patient.Gender.values[0]
sites = ["Tibia", "Radius"]

for _site in sites:
    if _site == "Tibia":
        meas_number = dataset_patient.Tibia_measurement_number.values[0]
        patient_properties = [
            dataset_patient.Tibia_total_volumetric_bone_mineral_density_mg_HA_ccm,
            dataset_patient.Tibia_cortical_vBMD_mg_HA_ccm,
            dataset_patient.Tibia_trabecular_bone_volume_fraction,
            dataset_patient.Tibia_rel_cortical_thickness,
            dataset_patient.Tibia_da,
            dataset_patient.Tibia_yield_stress,
        ]
        dataset_properties = [
            dataset.Tibia_total_volumetric_bone_mineral_density_mg_HA_ccm,
            dataset.Tibia_cortical_vBMD_mg_HA_ccm,
            dataset.Tibia_trabecular_bone_volume_fraction,
            dataset.Tibia_rel_cortical_thickness,
            dataset.Tibia_da,
            dataset.Tibia_yield_stress,
        ]

    elif _site == "Radius":
        meas_number = dataset_patient.Radius_measurement_number.values[0]
        patient_properties = [
            dataset_patient.Radius_total_volumetric_bone_mineral_density_mg_HA_ccm,
            dataset_patient.Radius_cortical_vBMD_mg_HA_ccm,
            dataset_patient.Radius_trabecular_bone_volume_fraction,
            dataset_patient.Radius_rel_cortical_thickness,
            dataset_patient.Radius_da,
            dataset_patient.Radius_yield_stress,
        ]
        
        dataset_properties = [
            dataset.Radius_total_volumetric_bone_mineral_density_mg_HA_ccm,
            dataset.Radius_cortical_vBMD_mg_HA_ccm,
            dataset.Radius_trabecular_bone_volume_fraction,
            dataset.Radius_rel_cortical_thickness,
            dataset.Radius_da,
            dataset.Radius_yield_stress,
        ]

    t_scores_patient = []
    for i in range(len(dataset_properties)):
        ref_avg_i, ref_std_i = dataset.__avg_std__(
            dataset_properties[i], patient_gender
        )
        t_score_i = db_stats.t_score(patient_properties[i], ref_avg_i, ref_std_i)
        t_scores_patient.append(t_score_i)

    db_stats.get_radar_chart(t_scores_patient, patient_UID)

    # * THIS IS AN EXAMPLE OF HOW TO CREATE SCATTER PLOTS
    results_table = pd.DataFrame()
    for i in range(len(dataset_properties)):
        res = db_stats.patient_assessment(
            dataset_patient.Age,
            patient_properties[i],
            dataset.Age,
            dataset_properties[i],
            dataset.__avg_std__(dataset_properties[i], patient_gender),
            PAPER=True,
            QUADRATIC=True,
        )
        results_table = pd.concat([results_table, res], axis=0)
    results_table = results_table.reset_index(drop=True)
    results_table["Property"] = [
        "\\quad \\acrshort{ttvbmd} [\\mgHAccm]",
        "\\quad \\acrshort{ctvbmd} [\\mgHAccm]",
        "\\quad \\acrshort{tb_bvtv} [-]",
        "\\quad \\acrshort{relctth} [-]",
        "\\quad \\acrshort{tb_da} [-]",
        "\\quad \\acrshort{app_ys} [\\SI{}{\mega\pascal}]",
    ]
    # move measurement to the first column
    cols = results_table.columns.tolist()
    cols = [cols[-1]] + cols[:-1]
    results_table = results_table[cols]
    print(results_table.to_latex(index=False, escape=False))

Check intersection at 37: 265.777966101695==265.7779661017152
Check slope at 37: -2.4291679778798425e-13
Reference mean:	265.777966101695
Reference std:	35.77648075345421
Quadratic fit:	 y(x) = -0.03922236317605657*x^2 + 2.902454875027943*x + 212.08255091370276
Check intersection at 37: 865.106779661017==865.1067796611272
Check slope at 37: 1.566746732351021e-12
Reference mean:	865.106779661017
Reference std:	32.034264330162976
Quadratic fit:	 y(x) = -0.11719771479446607*x^2 + 8.672630894792055*x + 704.6631081074452
Check intersection at 37: 0.28606779661016946==0.286067796610199
Check slope at 37: 1.43982048506075e-16
Reference mean:	0.28606779661016946
Reference std:	0.04374134964995134
Quadratic fit:	 y(x) = -3.5354267337430534e-05*x^2 + 0.0026162157829700034*x + 0.23766780462525125
Check intersection at 37: 0.06487753546051411==0.06487753546052026
Check slope at 37: 8.61940727125976e-18
Reference mean:	0.06487753546051411
Reference std:	0.011610398953949753
Quadratic fit:	 y(x) = -

/tmp/ipykernel_829057/3060090992.py:80: FutureWarning:

In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.



Check intersection at 37: 0.20891666666666664==0.20891666666670206
Check slope at 37: 4.852888924045118e-16
Reference mean:	0.20891666666666664
Reference std:	0.04496110999996974
Quadratic fit:	 y(x) = -2.6986937135473483e-05*x^2 + 0.001997033348025523*x + 0.1719715497282209
Check intersection at 37: 0.11251907158589543==0.11251907158590815
Check slope at 37: -7.795413620170777e-17
Reference mean:	0.11251907158589543
Reference std:	0.02142281351349384
Quadratic fit:	 y(x) = -1.1966475950126758e-05*x^2 + 0.0008855192203093021*x + 0.09613696601018752
Check intersection at 37: 1.822080462095578==1.8220804620956612
Check slope at 37: -8.087280845003875e-15
Reference mean:	1.822080462095578
Reference std:	0.10290124688792485
Quadratic fit:	 y(x) = -7.823230090885186e-05*x^2 + 0.005789190267246951*x + 1.7149804421517423
Check intersection at 37: 13.656020564185253==13.656020564187996
Check slope at 37: 5.2131909900055007e-14
Reference mean:	13.656020564185253
Reference std:	3.959029262672261

/tmp/ipykernel_829057/3060090992.py:80: FutureWarning:

In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.

